# 📖 Notebook 3: Request Transformation & API Versioning

An API gateway doesn't just forward requests — it can **transform** them. This means modifying headers, rewriting URLs, and changing response formats, all without touching backend code.

We'll cover:
- 🚫 **BAD**: Clients handle everything (headers, versioning, metadata)
- ✅ **BETTER**: Backend services handle transformation (duplicated logic)
- 🏆 **BEST**: Gateway handles transformation (centralized, consistent)

## Learning Objectives

By the end of this notebook, you'll understand:
- What header injection is and why it matters
- How the gateway adds tracking headers (X-Request-ID, X-Real-IP)
- How API versioning works at the gateway level
- How request/response transformation simplifies backend services
- Why the gateway must **overwrite or clear** every header the backend trusts

## 🛠️ Setup

Make sure infrastructure is running:

```bash
cd 05-microservices/api-gateway
docker compose up -d --build
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import requests
import json

GATEWAY = "http://localhost:8080"

def show(response):
    """Print HTTP status, headers, and JSON body."""
    print(f"Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text[:200])

def show_headers(response):
    """Print interesting response headers."""
    interesting = ["X-API-Version", "X-Served-By", "X-Request-ID", "Content-Type"]
    print("Response headers:")
    for key in interesting:
        if key.lower() in {k.lower(): k for k in response.headers}:
            for k, v in response.headers.items():
                if k.lower() == key.lower():
                    print(f"  {k}: {v}")

# Quick health check
try:
    r = requests.get(f"{GATEWAY}/health", timeout=3)
    print(f"✅ API Gateway: {r.json()['status']}")
except Exception as e:
    print(f"❌ API Gateway not running: {e}")
    print("   Run: cd 05-microservices/api-gateway && docker compose up -d --build")

---

## 🚫 BAD: Clients Handle Everything

Without a gateway doing transformation, clients are responsible for:
- Adding tracking headers (request IDs, timestamps)
- Knowing which API version to request
- Handling different response formats from different services
- Managing authentication tokens for each service

```
┌──────────────────────────────────────────┐
│  Client Code (BAD)                        │
│                                           │
│  headers = {                              │
│    "X-Request-ID": generate_uuid(),       │  ← Client generates tracking
│    "X-Client-Time": str(time.time()),     │  ← Client adds timestamps
│    "X-API-Version": "2",                  │  ← Client manages versioning
│    "Authorization": get_token(),           │  ← Client handles auth
│  }                                        │
│                                           │
│  # Different URL per service version!     │
│  if api_version == 1:                     │
│      url = "http://users-v1:5000"         │
│  elif api_version == 2:                   │
│      url = "http://users-v2:5001"         │
└──────────────────────────────────────────┘
```

This is messy, duplicated across every client, and breaks when anything changes.

In [ ]:
# BAD: Client has to add all metadata headers manually

import uuid
import time

print("🚫 BAD: Client manages its own headers")
print("=" * 55)
print()

# The client has to generate and manage all of this
client_headers = {
    "X-Request-ID": str(uuid.uuid4()),     # Client generates tracking ID
    "X-Client-Time": str(time.time()),      # Client adds timestamp
    "X-API-Version": "1",                   # Client specifies version
}

print("Headers the client has to create manually:")
for k, v in client_headers.items():
    print(f"  {k}: {v}")
print()

# Call the service directly — it receives only what the client sends
r = requests.get("http://localhost:5001/debug/headers", headers=client_headers)
backend_headers = r.json()["received_headers"]

print("What the backend received:")
for k, v in backend_headers.items():
    if k.startswith("X-"):
        print(f"  {k}: {v}")

print()
print("⚠️  Problems:")
print("   1. Every client must generate request IDs (inconsistent formats)")
print("   2. No trustworthy client IP once ANY proxy, CDN or load balancer sits")
print("      in front: the socket address is the proxy's, and the header the")
print("      backend would read instead is whatever the client chose to send")
print("   3. Clients can fake headers -- every one of the three above was")
print("      invented by the client, so the backend can trust none of them")
print("   4. Version management is scattered across all clients")
print()
print("   Notice: the backend echoed back EXACTLY what we sent, including the")
print("   request ID we made up. Hold that thought -- we come back to it below.")

# The BAD case is "the backend believes the client". Prove it.
assert backend_headers.get("X-Request-Id") == client_headers["X-Request-ID"], (
    "the direct-call backend should receive the client's own (forgeable) request ID")

---

## ✅ BETTER: Backend Services Handle Transformation

Instead of clients doing everything, each backend service could handle its own transformation — generating request IDs, checking versions, etc.

But this means **every service duplicates the same logic**:

```python
# In EVERY service (user_service.py, order_service.py, payment_service.py...)

@app.before_request
def add_request_metadata():
    # Generate request ID if not present
    if "X-Request-ID" not in request.headers:
        request.request_id = str(uuid.uuid4())
    
    # Check API version
    version = request.headers.get("X-API-Version", "1")
    request.api_version = version
    
    # Log the request
    log_request(request)
```

This works but creates maintenance headaches:
- Change the request ID format? Update **every** service
- Add a new header? Deploy **every** service
- Services implemented in different languages? Rewrite the logic in each one

In [ ]:
# BETTER: Each service adds its own metadata
# Our Flask services already do some of this (e.g., served_by field)

print("✅ BETTER: Backend services add their own metadata")
print("=" * 55)
print()

# Call user service directly — it adds 'served_by' but nothing else
print("User Service response (direct call):")
r = requests.get("http://localhost:5001/users/1")
show(r)
print()

# Call order service directly — same pattern, duplicated logic
print("Order Service response (direct call):")
r = requests.get("http://localhost:5002/orders/101")
show(r)
print()

print("⚠️  Both services independently add 'served_by'. Now imagine:")
print("   - User service is Python, Order service is Go, Payment is Java")
print("   - Each has to implement the same request ID generation")
print("   - Each has to implement the same versioning logic")
print("   - When you change the format, you deploy 3 services instead of 1")

---

## 🏆 BEST: Gateway Handles All Transformation

The API gateway can transform requests and responses in one centralized place.

### What Our Gateway Does

```nginx
location /api/users {
    # HEADER INJECTION — gateway adds these to every request
    proxy_set_header Host $host;
    proxy_set_header X-Real-IP $remote_addr;          # Real client IP
    proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;  # Proxy chain
    proxy_set_header X-Request-ID $request_id;        # Unique tracking ID
    
    proxy_pass http://user_backend/users;
}
```

The backend receives these headers **automatically** — no client or backend code needed.

In [ ]:
# BEST: Header Injection — see what the gateway adds

import time
time.sleep(1)  # Brief pause to avoid rate limiting from previous notebook

print("🏆 BEST: Gateway Header Injection")
print("=" * 55)
print()

# The /api/debug/headers endpoint shows what headers the backend received
# We send a bare request — no custom headers from the client
print("Client sends: GET /api/debug/headers (no custom headers)")
print()

r = requests.get(f"{GATEWAY}/api/debug/headers")
backend_headers = r.json()["received_headers"]

print("Headers the backend RECEIVED (injected by the gateway):")
print("-" * 55)

# Highlight the gateway-injected headers
gateway_headers = ["X-Real-Ip", "X-Forwarded-For", "X-Request-Id", "X-Gateway-Timestamp"]

for key, value in backend_headers.items():
    if key in gateway_headers:
        print(f"  🏷️  {key}: {value}  ← INJECTED by gateway")
    elif key in ["Host", "Connection"]:
        print(f"  📋 {key}: {value}")
    else:
        print(f"     {key}: {value}")

print()
print("💡 The client sent NO custom headers, but the backend got:")
print("   - X-Real-IP: the client's actual IP address")
print("   - X-Forwarded-For: the full proxy chain")
print("   - X-Request-ID: a unique ID for tracing this request")
print("   - X-Gateway-Timestamp: when the gateway received the request")

In [ ]:
# Each request gets a UNIQUE X-Request-ID — essential for debugging

print("🔍 Unique Request IDs (for distributed tracing)")
print("=" * 55)
print()
print("Every request through the gateway gets a unique tracking ID.")
print("This is how you trace a request across multiple services.")
print()

request_ids = []
for i in range(5):
    r = requests.get(f"{GATEWAY}/api/debug/headers")
    req_id = r.json()["received_headers"].get("X-Request-Id", "N/A")
    request_ids.append(req_id)
    print(f"  Request {i+1}: X-Request-ID = {req_id}")
    time.sleep(0.3)

# Verify all IDs are unique
unique = len(set(request_ids)) == len(request_ids)
print()
print(f"All IDs unique: {'✅ Yes' if unique else '❌ No'}")

# A trace ID that repeats is worse than no trace ID -- it merges unrelated
# requests into one "trace". Fail the notebook rather than print a sad face.
assert unique, f"gateway reused a request ID: {request_ids}"
assert all(i != "N/A" for i in request_ids), "gateway did not inject X-Request-ID at all"
print()
print("💡 In production, you'd log this ID everywhere:")
print("   Gateway log: [abc123] Routing to user-service")
print("   User service log: [abc123] Fetching user 42")
print("   Database log: [abc123] SELECT * FROM users WHERE id=42")
print("   → One ID connects the entire request journey!")

---

## 🛡️ The gateway is a trust boundary, not just a decorator

Header injection is where a gateway quietly becomes a **security control**, and
where it is easiest to get wrong.

The backend has no way to tell a header the gateway wrote from a header the
client sent — by the time the request arrives they are just bytes in the same
list. So for every header the backend treats as a fact, the gateway must do one
of two things:

| Directive | Effect | Safe? |
|-----------|--------|-------|
| `proxy_set_header X-Real-IP $remote_addr;` | **overwrite** with a value nginx computed | ✅ |
| `proxy_set_header X-Authenticated "";` | **clear** — nginx sends no such header at all | ✅ |
| *(no directive)* | client's value is forwarded verbatim | ❌ **forgeable** |

That last row is the trap: nginx passes unknown client headers through by
default, so a header you merely *forgot* to mention in a `location` block is a
header the client gets to write.

Concretely: `/api/auth/users` sets `X-Authenticated: true` after checking the API
key. If `/api/users` — which checks nothing — did not explicitly clear that
header, a client could send `X-Authenticated: true` to the open route and the
backend would see a request indistinguishable from an authenticated one. That is
why every non-auth `location` in our `nginx.conf` carries an explicit
`proxy_set_header X-Authenticated "";`.

Let's try to forge all three and see what survives.

In [ ]:
# Try to forge the headers the backend trusts. The gateway should win.

time.sleep(1)

forged = {
    "X-Authenticated": "true",          # "I already passed auth" -- a pure lie
    "X-Real-IP":       "10.0.0.1",      # pretend to be an internal caller
    "X-Request-ID":    "attacker-chosen-trace-id",
    "X-Forwarded-For": "10.0.0.1",      # pretend to be behind a trusted proxy
}

print("🛡️  Forged headers sent BY THE CLIENT to /api/debug/headers")
print("=" * 60)
for k, v in forged.items():
    print(f"  {k}: {v}")
print()

r = requests.get(f"{GATEWAY}/api/debug/headers", headers=forged)
got = {k.lower(): v for k, v in r.json()["received_headers"].items()}

print("What the BACKEND actually received:")
print("-" * 60)
for name, sent in forged.items():
    value = got.get(name.lower())
    if value is None:
        verdict = "🧹 CLEARED by the gateway (never sent to the backend)"
    elif value == sent:
        verdict = "☠️  PASSED THROUGH UNCHANGED -- forgeable!"
    else:
        verdict = "✅ REWRITTEN by the gateway"
    print(f"  {name:<16} = {str(value):<34} {verdict}")

print()
print("💡 X-Forwarded-For is the interesting one. `$proxy_add_x_forwarded_for`")
print("   APPENDS the peer address to whatever the client sent, because that is")
print("   what the standard says a proxy chain looks like. The attacker's fake")
print("   entry is still in there -- it just isn't the last one. Rule: when you")
print("   read XFF, count from the RIGHT, and only trust as many entries as you")
print("   have proxies you actually control.")

# These are the security properties of the config, not cosmetics.
assert "x-authenticated" not in got, (
    "a client forged X-Authenticated onto an unauthenticated route -- the "
    "location block is missing `proxy_set_header X-Authenticated \"\";`")
assert got.get("x-real-ip") != "10.0.0.1", (
    "the gateway let the client dictate X-Real-IP; it must overwrite it with $remote_addr")
assert got.get("x-request-id") != "attacker-chosen-trace-id", (
    "the gateway let the client dictate the trace ID")
assert got.get("x-forwarded-for", "").endswith(got.get("x-real-ip", "\x00")), (
    "the real peer address must be the LAST entry in X-Forwarded-For")

---

## 🏷️ API Versioning at the Gateway

APIs evolve over time. You might need to:
- Add new fields to responses
- Change the response structure
- Deprecate old endpoints

The gateway can handle versioning **without changing backend code**:

```nginx
# API v2 endpoint — gateway injects version header
location /api/v2/users {
    proxy_set_header X-API-Version "2";     # Tell backend: use v2 format
    add_header X-API-Version "2" always;     # Tell client: this is v2
    proxy_pass http://user_backend/users;
}
```

The backend checks the `X-API-Version` header and returns the appropriate format:

```
GET /api/users     →  v1 response (basic)
GET /api/v2/users  →  v2 response (with _metadata)
```

Both hit the **same backend service** — the gateway just tells it which version to use.

In [ ]:
# API Versioning — same backend, different response formats

time.sleep(1)

print("🏷️ API Versioning at the Gateway")
print("=" * 55)
print()

# V1: Basic response
print("📋 API v1: GET /api/users/1")
r1 = requests.get(f"{GATEWAY}/api/users/1")
show(r1)
print()

# V2: Enhanced response with metadata
print("📋 API v2: GET /api/v2/users/1")
r2 = requests.get(f"{GATEWAY}/api/v2/users/1")
show(r2)
print()

# Compare
v1_keys = set(r1.json().keys())
v2_keys = set(r2.json().keys())
new_keys = v2_keys - v1_keys

print("📊 Comparison:")
print(f"  v1 fields: {sorted(v1_keys)}")
print(f"  v2 fields: {sorted(v2_keys)}")
if new_keys:
    print(f"  New in v2:  {sorted(new_keys)}")
print()

print("💡 Both versions hit the SAME backend service.")
print("   The gateway just injects 'X-API-Version: 2' header for v2 requests.")
print("   Old clients keep using /api/users (v1) — nothing breaks!")

In [ ]:
# Check the response headers for v2 — gateway adds version info

time.sleep(0.5)

print("🔍 Response Headers: v1 vs v2")
print("=" * 55)
print()

r1 = requests.get(f"{GATEWAY}/api/users/1")
r2 = requests.get(f"{GATEWAY}/api/v2/users/1")

print("v1 response headers:")
for key in ["X-API-Version", "X-Served-By"]:
    val = r1.headers.get(key, "(not set)")
    print(f"  {key}: {val}")
print()

print("v2 response headers:")
for key in ["X-API-Version", "X-Served-By"]:
    val = r2.headers.get(key, "(not set)")
    print(f"  {key}: {val}")

print()
print("💡 The gateway adds response headers too! The client can check")
print("   X-API-Version to confirm which version they received.")

---

## 🔄 URL Rewriting (Path Transformation)

The gateway also transforms URL paths. Our clients use clean, versioned URLs, but the backends have simpler internal paths:

```
What the client sends:          What the backend receives:
────────────────────           ──────────────────────────
GET /api/users/1           →   GET /users/1
GET /api/orders/101        →   GET /orders/101
GET /api/v2/users/1        →   GET /users/1  (+ X-API-Version: 2)
GET /api/auth/users        →   GET /users    (+ X-Authenticated: true)
```

This is **path stripping** — the gateway removes the `/api` prefix before forwarding.  
The backend services don't even know they're behind a gateway!

In [ ]:
# URL Rewriting demo — same backend, many gateway paths

time.sleep(0.5)

print("🔄 URL Rewriting (Path Transformation)")
print("=" * 55)
print()
print("The gateway rewrites URLs before forwarding to backends:")
print()

rewrites = [
    ("/api/users",       "Open users endpoint"),
    ("/api/v2/users",    "Versioned users endpoint"),
    ("/api/auth/users",  "Authenticated users endpoint"),
]

for path, description in rewrites:
    headers = {}
    if "auth" in path:
        headers["X-API-Key"] = "demo-key-123"
    
    r = requests.get(f"{GATEWAY}{path}", headers=headers)
    if r.status_code == 200:
        data = r.json()
        has_meta = "_metadata" in data
        print(f"  {path:<25} → {description}")
        print(f"    Status: {r.status_code}, Users: {data.get('count', '?')}, Has metadata: {has_meta}")
    else:
        print(f"  {path:<25} → Status: {r.status_code}")
    time.sleep(0.3)

print()
print("💡 All three paths go to the SAME user service backend.")
print("   The gateway handles auth, versioning, and path rewriting.")
print("   The backend just serves users — simple and focused.")

---

## 🏗️ Putting It All Together

Here's the full picture of what our API gateway does for every request:

```
Client Request
    │
    ▼
┌───────────────────────────────────────────────────┐
│                  API GATEWAY                       │
│                                                   │
│  1. Rate Limiting     → Block if over limit (429) │
│  2. Authentication    → Block if no API key (401) │
│  3. Path Routing      → Pick the right backend    │
│  4. Header Injection  → Add X-Request-ID, etc.    │
│  5. URL Rewriting     → /api/users → /users       │
│  6. Load Balancing    → Pick a healthy instance    │
│  7. Response Headers  → Add X-API-Version, etc.   │
│                                                   │
└───────────────────────┬───────────────────────────┘
                        │
                        ▼
                  Backend Service
            (simple, focused, unaware)
```

All of this is configured in **one nginx config file**. No code changes to any backend service.

In [ ]:
# Full demo: trace a request through all gateway transformations

time.sleep(1)

print("🏗️ Full Request Journey Through the Gateway")
print("=" * 60)
print()

# Send an authenticated v2 request through the debug endpoint
print("Client sends:")
print("  GET http://localhost:8080/api/debug/headers")
print("  Headers: (none — just a bare request)")
print()

r = requests.get(f"{GATEWAY}/api/debug/headers")
data = r.json()
backend_headers = data["received_headers"]

print("Gateway transformations applied:")
print("-" * 60)

transformations = [
    ("URL Rewriting",    "/api/debug/headers → /debug/headers"),
    ("Host Header",      f"Host: {backend_headers.get('Host', '?')}"),
    ("Client IP",        f"X-Real-Ip: {backend_headers.get('X-Real-Ip', '?')}"),
    ("Proxy Chain",      f"X-Forwarded-For: {backend_headers.get('X-Forwarded-For', '?')}"),
    ("Request Tracking", f"X-Request-Id: {backend_headers.get('X-Request-Id', '?')}"),
    ("Timestamp",        f"X-Gateway-Timestamp: {backend_headers.get('X-Gateway-Timestamp', '?')}"),
    ("Load Balancing",   f"Routed to: {data.get('served_by', '?')}"),
]

for name, detail in transformations:
    print(f"  ✅ {name:<20} {detail}")

print()
print("💡 The client sent a bare GET request. The gateway added 5+ headers,")
print("   rewrote the URL, and load-balanced to a healthy instance.")
print("   Zero effort from the client, zero code in the backend.")

## 📚 Summary

### What We Learned

| Transformation | 🚫 BAD (Client) | ✅ BETTER (Backend) | 🏆 BEST (Gateway) |
|---------------|:---:|:---:|:---:|
| Request IDs | Client generates (forgeable) | Each service generates | Gateway generates |
| IP Tracking | Client-supplied, unverifiable | Not reliable | Gateway overwrites with the real peer address |
| Trust headers | Anything the client says | Anything the client says | Overwritten or cleared at the gateway |
| API Versioning | Client manages URLs | Each service checks | Gateway injects header |
| URL Rewriting | N/A | N/A | Gateway strips prefixes |
| Response Headers | N/A | Each service adds | Gateway adds |

### Key Takeaways

1. **Header injection** — the gateway adds tracking headers (X-Request-ID, X-Real-IP) to every request automatically
2. **The gateway is a trust boundary** — the backend cannot tell a gateway-written header from a client-written one, so every header the backend trusts must be explicitly *overwritten* (`proxy_set_header X-Real-IP $remote_addr;`) or *cleared* (`proxy_set_header X-Authenticated "";`). A header you forget to mention is a header the client controls
3. **API versioning** — different URL paths map to the same backend with different version headers
4. **URL rewriting** — clean public URLs (/api/users) map to simple internal paths (/users)
5. **Centralized transformation** — change once at the gateway, applies to all services
6. **Backend simplicity** — services focus on business logic, not cross-cutting concerns

### Interview Tip

> In system design interviews, don't spend too long on the API gateway. Mention it does *"routing, rate limiting, and basic middleware like auth and request tracking"* and move on. The gateway is important infrastructure, but interviewers want to see you design the core system.

### The Complete API Gateway Pattern

Across these three notebooks, we've covered the full API gateway responsibility set:

| Notebook | Capability | What It Does |
|----------|-----------|-------------|
| 1 | **Routing** | Maps URL paths to backend services |
| 1 | **Load Balancing** | Distributes traffic across instances |
| 1 | **Health Checks** | Passively ejects a backend after real requests to it fail (`max_fails`); active probing needs nginx Plus / Envoy |
| 2 | **Rate Limiting** | Prevents abuse, protects backends |
| 2 | **Authentication** | Validates API keys before forwarding |
| 3 | **Header Injection** | Adds tracking and metadata headers |
| 3 | **Trust Boundary** | Overwrites or clears headers the client must not control |
| 3 | **API Versioning** | Routes version-specific requests |
| 3 | **URL Rewriting** | Transforms public URLs to internal paths |

All of this in a single nginx config file. That's the power of an API gateway! 🎉